# Aula 10 — Limpeza, normalização e pipeline de scraping

Nas Aulas 8 e 9 vocês aprenderam a coletar: `requests` + BeautifulSoup para páginas estáticas, Playwright para páginas que só existem depois do JavaScript rodar. O que sai dessas coletas é sempre **dado bruto**: exatamente como veio do site, com toda a sujeira que uma página real carrega (espaço sobrando, capitalização que muda de item pra item, data escrita de um jeito diferente em cada linha, preço que é texto e não número, item repetido, campo vazio).

Hoje a coleta já está pronta (é o CSV em `dados/raw/`) e o trabalho é outro: transformar esse bruto numa base confiável, com um **pipeline reexecutável**: um roteiro de passos que você pode rodar de novo, com dado novo, e que sempre produz o mesmo tipo de resultado, documentando o que fez com cada problema encontrado no caminho.

É a última peça do Módulo 3, e fecha todo o pipeline de dados: coleta (Aula 8 ou 9) + limpeza (hoje) + saída tratada + log + README de reprodução.

## 1. Revisão rápida

Duas coisas voltam hoje, das Aulas 8 e 9: a diferença entre coletar com `requests` (página estática, HTML já vem pronto) e com Playwright (página dinâmica, precisa de navegador rodando JavaScript), e o fato de que qualquer coleta, seja qual for a ferramenta, chega com campos em formato de texto, tipos inconsistentes de linha pra linha e, às vezes, valor ausente. Hoje é sobre o que fazer com isso **depois** que a coleta terminou.

## 2. `dados/raw` e `dados/processed`: duas pastas, dois papéis

O pipeline de hoje trabalha com duas pastas dentro de `dados/`:

- **`dados/raw/`**: o que foi coletado, intocado. Nunca se escreve por cima de um arquivo aqui. É a fonte, a prova de onde os dados vieram.
- **`dados/processed/`**: o resultado depois de limpar, normalizar e validar. É o que se usa para analisar, gerar gráfico, alimentar outro script.

Por que separar assim, em vez de simplesmente limpar o CSV bruto no lugar? Duas razões práticas:

1. **Auditoria.** Se uma etapa de limpeza tiver um bug (por exemplo, um `.str.title()` aplicado na coluna errada), dá pra comparar o processado com o bruto e ver exatamente o que mudou. Se o bruto já tiver sido sobrescrito, essa comparação não existe mais.
2. **Reprocessamento.** Uma regra de limpeza muda (por exemplo, você decide tratar data ausente de outro jeito). Com o bruto intacto, é só rodar o pipeline de novo. Sem ele, a única fonte que sobrou já está contaminada pela decisão antiga.

**ATENÇÃO:** o erro mais comum desta aula não é técnico, é de hábito: salvar o resultado limpo com o mesmo nome do arquivo bruto, ou dentro da própria pasta `raw/`. A Seção 15 volta nisso.

## 3. Carregando o CSV bruto

`dados/raw/livros-coleta-bruta.csv` simula uma coleta de `books.toscrape.com` (o mesmo catálogo de prática da Aula 8) feita em datas diferentes e por pessoas diferentes, por isso a sujeira varia de linha pra linha: é exatamente o tipo de coisa que aparece numa coleta de verdade, só que concentrada num único arquivo pequeno para dar pra praticar.

In [1]:
import pandas as pd  # biblioteca de tabelas, já usada desde a Aula 4
from pathlib import Path  # para lidar com caminhos de arquivo sem escrever tudo na mão

caminho_bruto = Path("dados/raw/livros-coleta-bruta.csv")  # o arquivo intocado, exatamente como foi "coletado"
df_bruto = pd.read_csv(caminho_bruto)  # lê o CSV bruto como tabela; df_bruto nunca é alterado dali pra frente

linhas_brutas = len(df_bruto)  # guarda o total de linhas do bruto, para o log no fim da aula
print(f"Linhas no bruto: {linhas_brutas}")
df_bruto.head(10)  # espia as primeiras linhas

Linhas no bruto: 28


,titulo,categoria,preco,avaliacao,data_coleta,link
0,A Light in the Attic,Poetry,£51.77,3.0,2026-08-20,http://books.toscrape.com/catalogue/a-light-in...
1,Tipping the Velvet,Historical Fiction,£53.74,1.0,20/08/2026,http://books.toscrape.com/catalogue/tipping-th...
2,Soumission,fiction,£50.10,1.0,2026-08-20,http://books.toscrape.com/catalogue/soumission...
3,Sharp Objects,MYSTERY,£47.82,4.0,21-08-2026,http://books.toscrape.com/catalogue/sharp-obje...
4,Sapiens: A Brief History of Humankind,History,£54.23,5.0,2026-08-21,http://books.toscrape.com/catalogue/sapiens-a-...
5,The Requiem Red,young adult,£22.65,NaN,22/08/2026,http://books.toscrape.com/catalogue/the-requie...
6,The Dirty Little Secrets of Getting Your Dream...,Business,£33.34,4.0,2026-08-22,http://books.toscrape.com/catalogue/the-dirty-...
7,The Coming Woman: A Novel Based on the Life of...,Biography,£17.93,3.0,22-Aug-2026,http://books.toscrape.com/catalogue/the-coming...
8,The Boys in the Boat,Default,£22.60,4.0,2026-08-23,http://books.toscrape.com/catalogue/the-boys-i...
9,The Black Maria,Poetry,NaN,1.0,2026-08-23,http://books.toscrape.com/catalogue/the-black-...


**O que observar:** já dá pra ver sujeira nas primeiras linhas: espaço sobrando antes de "Tipping the Velvet", `MYSTERY` todo maiúsculo enquanto `fiction` está todo minúsculo, preço com `£` na frente. Isso não é erro de leitura do Pandas, é assim mesmo que o arquivo bruto está, de propósito.

## 4. Conferir tipos e valores ausentes antes de mexer

Antes de limpar qualquer coisa, vale entender o tamanho do problema: quais colunas o Pandas leu como quê, e quantos valores ausentes cada uma tem. `df.dtypes` e `df.isna().sum()` já apareceram na Aula 5; aqui é a mesma ideia, usada como primeiro diagnóstico de um pipeline.

In [2]:
print(df_bruto.dtypes)  # tipo de cada coluna; preço e data devem aparecer como "object" (texto), não número/data
print()
print(df_bruto.isna().sum())  # quantos valores ausentes por coluna, antes de qualquer limpeza

titulo             str
categoria          str
preco              str
avaliacao      float64
data_coleta        str
link               str
dtype: object

titulo         1
categoria      0
preco          1
avaliacao      1
data_coleta    0
link           0
dtype: int64


**O que observar:** `preco` e `data_coleta` vêm como texto (`object`), mesmo devendo ser número e data. Isso é normal em CSV: o Pandas só reconhece um tipo mais específico se o formato for uniforme o bastante para adivinhar sozinho, e aqui não é (símbolo de moeda, formatos de data variados). Corrigir esses tipos é boa parte do trabalho de hoje.

## 5. Trabalhando numa cópia

A partir daqui, todo o processamento acontece em `df`, uma cópia de `df_bruto`. `df_bruto` continua existindo, sem nenhuma alteração, até o fim do notebook: é o equivalente, dentro do próprio Python, de nunca sobrescrever o arquivo em `dados/raw/`.

In [3]:
df = df_bruto.copy()  # cópia independente; mexer em df não afeta df_bruto
print(f"Linhas para processar: {len(df)}")

Linhas para processar: 28


## 6. Duplicatas: primeiro passo do pipeline, não última escolha

`duplicated()` e `drop_duplicates()` já apareceram na Aula 5 (revise lá se o comando ainda não é familiar). A diferença aqui é de postura: numa exploração pontual, tirar duplicata é uma decisão que você toma quando percebe o problema. Num **pipeline**, é um passo obrigatório, sempre no mesmo lugar da sequência, porque toda nova coleta pode trazer linha repetida (o mesmo item raspado duas vezes, por exemplo, se o script rodou parcialmente e foi reexecutado sem cuidado).

In [4]:
duplicatas_encontradas = df.duplicated().sum()  # conta quantas linhas são idênticas a uma linha anterior
print(f"Linhas duplicadas encontradas: {duplicatas_encontradas}")

df = df.drop_duplicates()  # mantém só a primeira ocorrência de cada linha repetida
linhas_apos_dedup = len(df)
print(f"Linhas após remover duplicatas: {linhas_apos_dedup}")

Linhas duplicadas encontradas: 3
Linhas após remover duplicatas: 25


## 7. Normalização de texto

Três problemas comuns em coluna de texto vindo de scraping: espaço sobrando nas pontas (o HTML original tinha indentação, ou alguém copiou/colou com espaço junto), capitalização inconsistente (`fiction`, `Fiction`, `MYSTERY` deveriam contar como a mesma categoria, mas hoje não contam) e, às vezes, duas formas de escrever completamente diferentes para o mesmo conceito (`Sci-Fi` e `Science Fiction` não têm nenhuma letra maiúscula em comum pra `.str.title()` resolver sozinho).

- `.str.strip()` remove espaço em branco do início e do fim do texto.
- `.str.title()` deixa a primeira letra de cada palavra maiúscula, o resto minúsculo (`"HISTORICAL fiction"` → `"Historical Fiction"`). Para campos onde faz mais sentido tudo minúsculo (por exemplo, algo que vai virar uma chave de comparação), `.str.lower()` é a opção.
- Quando duas grafias diferentes significam a mesma coisa (sinônimo, sigla, abreviação), `.str.title()`/`.str.lower()` não resolvem: isso exige uma tabela de equivalência feita à mão, porque só uma pessoa sabe que "Sci-Fi" e "Science Fiction" são a mesma categoria.
- E quando o texto vem inteiro em CAIXA ALTA (`"THE HOBBIT"`), `.str.title()` também resolve a capitalização, mas com uma ressalva: ele capitaliza *toda* palavra, inclusive artigos e preposições que o inglês normalmente deixa minúsculos (`"the lord of the rings"` viraria `"The Lord Of The Rings"`, não `"The Lord of the Rings"`). Por isso só vale aplicar em títulos que já estão em caixa alta, não em todos: senão você troca uma sujeira por outra.

In [5]:
df["titulo"] = df["titulo"].str.strip()  # remove espaço sobrando no início/fim do título

df["categoria"] = df["categoria"].str.strip().str.title()  # remove espaço e padroniza capitalização
# "poetry ", "POETRY" e "Poetry" viram todos "Poetry": agora dá pra agrupar/contar por categoria de verdade

categorias_equivalentes = {"Sci-Fi": "Science Fiction"}  # mapeamento manual: grafias diferentes, mesmo conceito
categorias_unificadas = df["categoria"].isin(categorias_equivalentes.keys()).sum()  # quantas linhas usavam a grafia alternativa
df["categoria"] = df["categoria"].replace(categorias_equivalentes)  # troca "Sci-Fi" por "Science Fiction"
print(f"Categorias unificadas por nome equivalente: {categorias_unificadas}")

titulos_caixa_alta = df["titulo"].str.isupper()  # True para título inteiro em maiúsculas (ex.: "THE HOBBIT")
titulos_normalizados = titulos_caixa_alta.sum()
df.loc[titulos_caixa_alta, "titulo"] = df.loc[titulos_caixa_alta, "titulo"].str.title()  # só normaliza esses
print(f"Títulos em caixa alta normalizados: {titulos_normalizados}")

df[["titulo", "categoria"]].head(10)

Categorias unificadas por nome equivalente: 1
Títulos em caixa alta normalizados: 1


,titulo,categoria
0,A Light in the Attic,Poetry
1,Tipping the Velvet,Historical Fiction
2,Soumission,Fiction
3,Sharp Objects,Mystery
4,Sapiens: A Brief History of Humankind,History
5,The Requiem Red,Young Adult
6,The Dirty Little Secrets of Getting Your Dream...,Business
7,The Coming Woman: A Novel Based on the Life of...,Biography
8,The Boys in the Boat,Default
9,The Black Maria,Poetry


**O que observar:** compare com a Seção 3. `"  Tipping the Velvet"` perdeu o espaço da frente, as categorias que estavam em maiúsculo ou minúsculo agora seguem o mesmo padrão, `"Sci-Fi"` virou `"Science Fiction"` (mesma categoria de `"Mesaerion..."`, agora unificada) e `"THE HOBBIT"` virou `"The Hobbit"`. Sem o primeiro passo, um `df.groupby("categoria")` contaria `"Poetry"` e `"poetry "` como categorias diferentes; sem o mapeamento manual, `"Sci-Fi"` e `"Science Fiction"` continuariam separadas mesmo sendo a mesma coisa — é exatamente o tipo de erro silencioso que esta aula existe pra evitar.

## 8. Normalização de datas

A coluna `data_coleta` mistura formatos: `2026-08-20` (ano-mês-dia), `20/08/2026` (dia/mês/ano), `21-08-2026`, `22-Aug-2026` (mês por extenso). Isso acontece na vida real quando a mesma coleta é feita por pessoas diferentes, em máquinas com configuração regional diferente, ou por scripts escritos em momentos diferentes.

`pd.to_datetime()` converte texto em data de verdade. Dois argumentos importam aqui:

- `dayfirst=True`: avisa que, quando o formato for ambíguo (`20/08/2026`), o primeiro número é o dia, não o mês (convenção usada no Brasil; sem isso, o Pandas assumiria mês primeiro, ao estilo americano).
- `errors="coerce"`: se um valor não for uma data válida de jeito nenhum (uma data com dia 35, por exemplo), em vez de quebrar o programa inteiro com erro, o Pandas troca aquele valor por `NaT` ("Not a Time", o equivalente de valor ausente para datas) e segue para a próxima linha.

In [6]:
df["data_coleta"] = pd.to_datetime(
    df["data_coleta"],
    format="mixed",  # aceita formatos diferentes linha a linha, em vez de exigir um único padrão fixo
    dayfirst=True,  # em caso de ambiguidade, o primeiro número do texto é o dia
    errors="coerce",  # data impossível de interpretar vira NaT, sem derrubar o processamento
)

datas_invalidas = df["data_coleta"].isna().sum()  # quantas linhas ficaram sem data válida
print(f"Datas que não converteram (viraram NaT): {datas_invalidas}")

df[["titulo", "data_coleta"]].head(10)

Datas que não converteram (viraram NaT): 2


,titulo,data_coleta
0,A Light in the Attic,2026-08-20
1,Tipping the Velvet,2026-08-20
2,Soumission,2026-08-20
3,Sharp Objects,2026-08-21
4,Sapiens: A Brief History of Humankind,2026-08-21
5,The Requiem Red,2026-08-22
6,The Dirty Little Secrets of Getting Your Dream...,2026-08-22
7,The Coming Woman: A Novel Based on the Life of...,2026-08-22
8,The Boys in the Boat,2026-08-23
9,The Black Maria,2026-08-23


**O que observar:** a coluna `data_coleta` agora é do tipo `datetime64`, não texto (confira com `df.dtypes` se tiver dúvida). Uma das linhas (`2026-08-35`, dia 35 não existe) virou `NaT`: o `errors="coerce"` fez exatamente o prometido, isolou o problema numa linha em vez de travar o notebook inteiro.

## 9. De "preço como texto" para número de verdade

`preco` chega como `"£51.77"`: um símbolo de moeda colado num número, tudo dentro de uma string. Sem converter, `df["preco"].mean()` nem roda (não dá pra tirar média de texto). O caminho tem duas etapas: primeiro tirar o que não é número da string (`.str.replace()`), depois converter o que sobrou para número de verdade.

`pd.to_numeric(..., errors="coerce")` funciona igual ao `pd.to_datetime` da seção anterior: o que não vira número (por exemplo, alguém digitou `"Grátis"` no lugar de um preço) vira `NaN`, sem quebrar o resto.

In [7]:
df["preco"] = df["preco"].str.replace("£", "", regex=False).str.strip()  # tira o símbolo de moeda e o espaço
df["preco"] = pd.to_numeric(df["preco"], errors="coerce")  # converte o que sobrou para número; o resto vira NaN

precos_invalidos = df["preco"].isna().sum()  # quantas linhas ficaram sem preço numérico (ausente ou "Grátis")
print(f"Preços ausentes ou não numéricos: {precos_invalidos}")

df[["titulo", "preco"]].head(10)

Preços ausentes ou não numéricos: 2


,titulo,preco
0,A Light in the Attic,51.77
1,Tipping the Velvet,53.74
2,Soumission,50.10
3,Sharp Objects,47.82
4,Sapiens: A Brief History of Humankind,54.23
5,The Requiem Red,22.65
6,The Dirty Little Secrets of Getting Your Dream...,33.34
7,The Coming Woman: A Novel Based on the Life of...,17.93
8,The Boys in the Boat,22.60
9,The Black Maria,NaN


**O que observar:** `preco` agora é `float64` (confira com `df.dtypes`). A linha que tinha `"Grátis"` e a que estava vazia no bruto viraram `NaN`: nenhuma das duas era um preço numérico, e forçar um valor ali seria inventar dado que a coleta não trouxe.

## 10. Valores ausentes: decisão explícita, não automática

Não existe uma resposta única para "o que fazer com um valor ausente". As opções mais comuns:

1. **Descartar a linha inteira**, quando o campo ausente é essencial e não dá pra seguir sem ele.
2. **Preencher com um valor padrão**, quando existe um valor que representa "não informado" sem se disfarçar de dado real.
3. **Deixar como está (`NaN`/`NaT`) e documentar**, quando inventar um valor seria pior do que admitir que ele está ausente.

O que importa não é qual dessas três você escolhe, é que a escolha apareça no código e no log, para quem usar o dado processado depois (inclusive você, daqui a três meses) entender o que aconteceu. Para o CSV de hoje:

- **`titulo` ausente → descarta a linha.** Não dá pra analisar, agrupar ou citar um livro sem título.
- **`avaliacao` ausente → preenche com `0`**, um valor que não existe na escala real de avaliação (1 a 5), então funciona como marcador de "sem avaliação registrada" sem se confundir com uma nota baixa de verdade.
- **`preco` ausente ou não numérico → mantém como `NaN`, documentado.** Inventar um preço distorceria qualquer média ou soma calculada depois.

In [8]:
linhas_antes_titulo = len(df)  # tamanho da tabela antes de descartar por título ausente
df = df.dropna(subset=["titulo"])  # descarta linha sem título: campo essencial, sem ele não dá pra seguir
linhas_descartadas_titulo = linhas_antes_titulo - len(df)
print(f"Linhas descartadas por título ausente: {linhas_descartadas_titulo}")

avaliacoes_preenchidas = df["avaliacao"].isna().sum()  # quantas linhas tinham avaliação ausente
df["avaliacao"] = df["avaliacao"].fillna(0).astype(int)  # preenche com 0 = "sem avaliação registrada"
print(f"Avaliações preenchidas com 0 (ausentes no bruto): {avaliacoes_preenchidas}")

precos_mantidos_ausentes = df["preco"].isna().sum()  # preço ausente/não numérico, mantido como está, por decisão
print(f"Preços mantidos como ausentes (decisão documentada): {precos_mantidos_ausentes}")

Linhas descartadas por título ausente: 1
Avaliações preenchidas com 0 (ausentes no bruto): 1
Preços mantidos como ausentes (decisão documentada): 2


## 11. Valores fora do intervalo esperado

Nem todo problema é ausência ou tipo errado: às vezes o valor está lá, no tipo certo, só que fora do que faz sentido pro domínio. A escala de avaliação do site vai de 1 a 5 estrelas; uma linha com `avaliacao = 7` não é um valor ausente nem um erro de conversão, é um valor que a coleta trouxe registrado errado (ou raspado do lugar errado na página).

Esse tipo de erro só aparece se alguém souber a regra de negócio (aqui, "avaliação vai de 1 a 5") e checar por ela explicitamente — nenhuma conversão de tipo ou `.isna()` pega isso sozinho. A decisão é a mesma lógica da Seção 10: documentar e manter como `NaN`, em vez de inventar um valor plausível (descartar a nota e "arredondar" pra 5, por exemplo, seria alterar um dado que a coleta realmente trouxe).

In [9]:
mask_fora_intervalo = (df["avaliacao"] != 0) & ((df["avaliacao"] < 1) | (df["avaliacao"] > 5))
# avaliacao == 0 é o marcador de "ausente" que a Seção 10 acabou de criar, não conta aqui de novo

avaliacoes_fora_intervalo = mask_fora_intervalo.sum()
df["avaliacao"] = df["avaliacao"].astype(float)  # precisa virar float pra aceitar NaN (int não aceita)
df.loc[mask_fora_intervalo, "avaliacao"] = float("nan")
print(f"Avaliações fora do intervalo 1-5 (mantidas como NaN, decisão documentada): {avaliacoes_fora_intervalo}")

Avaliações fora do intervalo 1-5 (mantidas como NaN, decisão documentada): 1


## 12. Validação: conferir antes de considerar pronto

Antes de salvar qualquer coisa em `dados/processed/`, vale checar que a tabela realmente tem o mínimo esperado: as colunas obrigatórias existem, e nenhuma delas está inteiramente vazia. Isso parece óbvio, mas é o tipo de checagem que evita salvar (e usar depois, em outro lugar) uma tabela quebrada sem perceber, por exemplo se um passo anterior renomeou uma coluna sem querer.

O bloco abaixo não é uma função, é só um laço com `if`/`raise`: por coluna obrigatória, confere se ela existe e se não está inteiramente vazia; se alguma condição falhar, `raise ValueError(...)` interrompe o notebook ali mesmo, com uma mensagem clara sobre o que faltou (em vez de deixar o problema aparecer, confuso, só lá na frente).

In [10]:
colunas_obrigatorias = ["titulo", "categoria", "preco", "avaliacao", "data_coleta"]  # o mínimo que este pipeline exige

for coluna in colunas_obrigatorias:  # percorre cada coluna obrigatória, uma de cada vez
    if coluna not in df.columns:  # a coluna nem existe na tabela
        raise ValueError(f"Coluna obrigatória ausente: {coluna}")
    if df[coluna].isna().all():  # a coluna existe, mas está inteiramente vazia (todo mundo é NaN)
        raise ValueError(f"Coluna obrigatória totalmente vazia: {coluna}")

print("Validação de colunas obrigatórias: OK")

Validação de colunas obrigatórias: OK


**O que observar:** se você quiser ver o `raise` funcionando de verdade, troque um nome em `colunas_obrigatorias` por algo que não existe (`"preco_final"`, por exemplo) e rode a célula de novo: o notebook para exatamente naquele ponto, com uma mensagem que diz qual coluna faltou, não um erro genérico lá na frente.

## 13. `try`/`except`: capturando erro sem derrubar o pipeline inteiro

Até aqui, as conversões de data e preço usaram `errors="coerce"`, um atalho pronto do Pandas que já faz, por baixo dos panos, algo parecido com o que vem agora. Mas vale entender o mecanismo geral, porque nem toda situação tem um atalho pronto como esse.

Uma **exceção** é o jeito do Python de avisar que algo deu errado no meio da execução (`ValueError`, `TypeError`, `KeyError`, e vários outros tipos, cada um para uma família de problema). Sem tratamento, uma exceção interrompe o programa inteiro na hora em que acontece. `try`/`except` muda isso: o que está dentro do `try` é tentado; se der uma exceção do tipo indicado no `except`, o programa não quebra, executa o bloco do `except` e segue em frente.

**ATENÇÃO:** sempre indique o tipo específico de exceção que você espera (`except ValueError:`), em vez de um `except:` genérico (que captura literalmente qualquer erro). Um `except:` sem tipo esconde até erro de digitação no seu próprio código (por exemplo, uma variável com nome errado, que gera `NameError`), fazendo o programa "engolir" um bug de verdade em silêncio, sem avisar. Capturar só o tipo esperado deixa esse tipo de bug aparecer normalmente, do jeito que deveria.

In [11]:
precos_brutos = ["51.77", "Grátis", "22.65", "abc"]  # simula preços já sem o símbolo £, um deles inválido

precos_convertidos = []  # guarda o resultado de cada tentativa de conversão
erros_de_conversao = []  # log dos valores que não converteram

for valor in precos_brutos:  # percorre cada preço, um de cada vez
    try:
        preco_numero = float(valor)  # tenta converter para número
        precos_convertidos.append(preco_numero)  # deu certo: guarda o número
    except ValueError:  # captura especificamente erro de conversão, não qualquer erro
        precos_convertidos.append(None)  # guarda None no lugar de derrubar o programa inteiro
        erros_de_conversao.append(valor)  # registra o valor problemático

print(f"Convertidos: {precos_convertidos}")
print(f"Valores que não converteram: {erros_de_conversao}")

Convertidos: [51.77, None, 22.65, None]
Valores que não converteram: ['Grátis', 'abc']


**O que observar:** o laço processa as quatro linhas sem travar em `"Grátis"` nem em `"abc"`, e ainda assim registra os dois como problema. É exatamente esse comportamento que `pd.to_numeric(errors="coerce")` e `pd.to_datetime(errors="coerce")` já entregam prontos, em uma coluna inteira de uma vez (por isso usamos eles nas Seções 8 e 9, em vez de escrever um laço manual). Mas quando você estiver processando algo linha a linha, sem um atalho vetorizado pronto do Pandas, `try`/`except` é a ferramenta geral para isso: capturar o erro esperado, registrar, seguir em frente.

## 14. Log: registrar o que o pipeline fez

Um log não precisa ser sofisticado: uma lista de linhas de texto, com números e decisões, já resolve. O valor de ter isso não é para hoje, é para daqui a um mês, quando alguém (você incluído) precisar entender por que o processado tem menos linhas que o bruto, sem reler todo o notebook.

In [12]:
log = []  # cada item vira uma linha do arquivo de log
log.append(f"Linhas no bruto: {linhas_brutas}")
log.append(f"Linhas duplicadas removidas: {duplicatas_encontradas}")
log.append(f"Linhas após remover duplicatas: {linhas_apos_dedup}")
log.append(f"Categorias unificadas por nome equivalente (ex.: Sci-Fi -> Science Fiction): {categorias_unificadas}")
log.append(f"Títulos em caixa alta normalizados: {titulos_normalizados}")
log.append(f"Linhas descartadas por título ausente: {linhas_descartadas_titulo}")
log.append(f"Datas inválidas (viraram NaT): {datas_invalidas}")
log.append(f"Preços ausentes ou não numéricos (mantidos como NaN, decisão documentada): {precos_mantidos_ausentes}")
log.append(f"Avaliações preenchidas com 0 (ausentes no bruto): {avaliacoes_preenchidas}")
log.append(f"Avaliações fora do intervalo 1-5 (mantidas como NaN, decisão documentada): {avaliacoes_fora_intervalo}")
log.append(f"Linhas no processado final: {len(df)}")

for linha in log:  # imprime o log inteiro, uma linha por vez
    print(linha)

Linhas no bruto: 28
Linhas duplicadas removidas: 3
Linhas após remover duplicatas: 25
Categorias unificadas por nome equivalente (ex.: Sci-Fi -> Science Fiction): 1
Títulos em caixa alta normalizados: 1
Linhas descartadas por título ausente: 1
Datas inválidas (viraram NaT): 2
Preços ausentes ou não numéricos (mantidos como NaN, decisão documentada): 2
Avaliações preenchidas com 0 (ausentes no bruto): 1
Avaliações fora do intervalo 1-5 (mantidas como NaN, decisão documentada): 1
Linhas no processado final: 24


## 15. Salvando o resultado em `dados/processed/`

Última etapa: gravar a tabela limpa num arquivo novo, dentro de `dados/processed/`, e o log junto com ela. `dados/raw/livros-coleta-bruta.csv` continua exatamente como estava no início do notebook, disponível para conferência ou reprocessamento.

In [13]:
Path("dados/processed").mkdir(parents=True, exist_ok=True)  # garante que a pasta existe antes de salvar

caminho_processado = Path("dados/processed/livros-tratado.csv")
df.to_csv(caminho_processado, index=False)  # salva a tabela limpa, sem a coluna de índice numérico
print(f"Processado salvo em: {caminho_processado}")

caminho_log = Path("dados/processed/log-pipeline.txt")
caminho_log.write_text("\n".join(log), encoding="utf-8")  # salva o log como texto simples, uma linha por item
print(f"Log salvo em: {caminho_log}")

print(f"Bruto continua intocado em: {caminho_bruto}")

Processado salvo em: dados/processed/livros-tratado.csv
Log salvo em: dados/processed/log-pipeline.txt
Bruto continua intocado em: dados/raw/livros-coleta-bruta.csv


**O que observar:** confira os dois arquivos novos na pasta `dados/processed/`. Abra `livros-tratado.csv` (num editor de texto, no VS Code, ou de volta com `pd.read_csv`) e confirme que `preco` e `data_coleta` aparecem como número e data, não mais como texto cru com `£` ou formato misturado. Abra também `log-pipeline.txt` e confira se os números batem com o que apareceu impresso na Seção 13.

## 16. Quando der errado

- **Sobrescrevi (ou reprocessei em cima d)o bruto sem querer.** É o erro mais comum e mais silencioso desta aula: um `df.to_csv("dados/raw/livros-coleta-bruta.csv")` por engano apaga a fonte original para sempre, sem aviso nenhum. Prevenção: nomeie a saída sempre com um nome diferente do bruto (`livros-tratado.csv`, não `livros-coleta-bruta.csv`), salve sempre em `dados/processed/`, nunca em `dados/raw/`, e, se possível, deixe `dados/raw/` fora de qualquer célula que use `.to_csv()`, `.to_excel()` ou similar.
- **`pd.to_datetime()` devolveu muito mais `NaT` do que o esperado.** Geralmente é ambiguidade de formato: confira se `dayfirst=True` está presente (sem ele, `20/08/2026` pode ser lido como mês 20, que não existe, e virar `NaT` por engano) e se `format="mixed"` está lá quando a coluna realmente mistura formatos diferentes.
- **`pd.to_numeric()` devolveu `NaN` numa linha que "parecia" número.** Sobrou algum caractere que não é dígito nem ponto decimal (espaço, símbolo de moeda que o `.str.replace()` não cobriu, vírgula como separador decimal em vez de ponto). Espie o valor bruto daquela linha antes de assumir que o comando está errado.
- **`ValueError: Unable to parse string` sem `errors="coerce"`.** Se você tirar o `errors="coerce"` de `pd.to_datetime()` ou `pd.to_numeric()`, a primeira linha problemática derruba a conversão inteira, em vez de virar `NaN`/`NaT`. Normalmente é isso que você quer evitar num pipeline (a menos que "parar tudo se algo estiver errado" seja mesmo a intenção).
- **`raise ValueError(...)` da Seção 11 interrompeu o notebook.** É o comportamento esperado quando falta uma coluna obrigatória ou ela está inteiramente vazia. Não é bug, é a validação fazendo o trabalho dela: confira o nome das colunas (`df.columns`) e os passos anteriores antes de simplesmente remover a checagem.
- **`ModuleNotFoundError: No module named 'pandas'`.** O ambiente não foi criado ou as dependências não foram instaladas. Rode `uv venv .venv` e `uv pip install -r requirements.txt` de novo, dentro da pasta certa (Seção 4 da Aula 8 tem o mesmo passo, em mais detalhe).

## 17. Prática: faça agora

Abra `exercicios/exercicio-10-limpeza-normalizacao-pipeline.ipynb`. Ele é o **Projeto 4**: pegue uma coleta sua (da Aula 8 ou da Aula 9), monte um pipeline completo (coleta → limpeza → validação → saída tratada em `data/processed` → log → README de reprodução), seguindo o mesmo roteiro de hoje.